# Board Game Recommender — Phase 1 (Trevor)

**Models implemented here:**
1. **Baseline**: Bayesian-weighted popularity
2. **Basic**: Content-Based Filtering (TF-IDF on game features)
3. **Advanced**: Two-Tower Neural Network

**Dataset**: BoardGameGeek Reviews (Kaggle) — `jvanelteren/boardgamegeek-reviews`

**Evaluation**: Leave-one-out per user (random, fixed seed — reviews CSV has no timestamps). Metrics: RMSE (rating prediction on full test set), Recall@10 and NDCG@10 (ranking on 10K-user subsample, 99 negatives per positive).

All three models share the same train/val/test split and evaluation harness so results are directly comparable with Alex's and Brandon's models.


## 1. Setup

In [1]:
# Colab: mount drive (optional, for caching) and install deps
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount skipped: {e}")

!pip install -q kaggle scikit-learn pandas numpy torch tqdm

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cpu':
    print("\n" + "!"*70)
    print("WARNING: Running on CPU. Two-Tower training will take HOURS.")
    print("Go to Runtime > Change runtime type > T4 GPU (free) and re-run.")
    print("!"*70 + "\n")

Device: cuda


## 2. Load Dataset

Download from Kaggle. Upload your `kaggle.json` API token to Colab first (Account → Create New API Token).

In [3]:
# Upload kaggle.json if not already present
if IN_COLAB and not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print("Upload your kaggle.json:")
    files.upload()
    !mkdir -p /root/.kaggle
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

# Download dataset
if not os.path.exists('boardgamegeek-reviews.zip'):
    !kaggle datasets download -d jvanelteren/boardgamegeek-reviews
    !unzip -q boardgamegeek-reviews.zip -d bgg_data

!ls bgg_data

Upload your kaggle.json:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/jvanelteren/boardgamegeek-reviews
License(s): other
100% 1.61G/1.61G [01:41<00:00, 17.0MB/s]

2020-08-19.csv	     bgg-19m-reviews.csv	  games_detailed_info.csv
2022-01-08.csv	     bgg-26m-reviews.csv
bgg-15m-reviews.csv  games_detailed_info2025.csv


In [4]:


# Load reviews and game metadata
# The dataset has multiple files; we use the reviews CSV and games details CSV.
DATA_DIR = 'bgg_data'

# Reviews: user, rating, comment, game id, name
reviews = pd.read_csv(f'{DATA_DIR}/bgg-19m-reviews.csv', usecols=['user', 'rating', 'ID', 'name'])
reviews = reviews.rename(columns={'ID': 'game_id'})
reviews = reviews.dropna(subset=['user', 'rating', 'game_id'])
reviews['rating'] = reviews['rating'].astype(float)
reviews = reviews[(reviews['rating'] >= 1) & (reviews['rating'] <= 10)]

# Game metadata for content-based model
games = pd.read_csv(f'{DATA_DIR}/games_detailed_info.csv', low_memory=False)
print(f"Reviews: {len(reviews):,}")
print(f"Games: {games['id'].nunique():,}")
print(f"Users: {reviews['user'].nunique():,}")

Reviews: 18,964,728
Games: 21,631
Users: 412,815


In [5]:
# Filter: keep users with >= 5 ratings and games with >= 20 ratings
# This removes extreme sparsity and makes evaluation meaningful.
MIN_USER_RATINGS = 5
MIN_GAME_RATINGS = 20

for _ in range(3):  # iterate until stable
    ucounts = reviews['user'].value_counts()
    gcounts = reviews['game_id'].value_counts()
    reviews = reviews[
        reviews['user'].isin(ucounts[ucounts >= MIN_USER_RATINGS].index) &
        reviews['game_id'].isin(gcounts[gcounts >= MIN_GAME_RATINGS].index)
    ]

print(f"After filtering: {len(reviews):,} reviews, {reviews['user'].nunique():,} users, {reviews['game_id'].nunique():,} games")

After filtering: 18,716,519 reviews, 272,319 users, 21,802 games


In [6]:
# Encode users and games to contiguous IDs
user_enc = LabelEncoder()
game_enc = LabelEncoder()
reviews['user_idx'] = user_enc.fit_transform(reviews['user'])
reviews['game_idx'] = game_enc.fit_transform(reviews['game_id'])

N_USERS = reviews['user_idx'].nunique()
N_GAMES = reviews['game_idx'].nunique()
print(f"N_USERS={N_USERS}, N_GAMES={N_GAMES}")

N_USERS=272319, N_GAMES=21802


## 3. Train/Val/Test Split

**Strategy**: Per-user leave-one-out for test, one more for validation. This dataset has no timestamps in the reviews CSV, so we use random per-user holdout with a fixed seed — matching the protocol commonly used on BGG and keeping things comparable across teammates.

If Alex and Brandon want timestamps, they can swap in a time-based split — the rest of the pipeline is agnostic.

In [7]:
def leave_one_out_split(df, seed=SEED):
    """For each user: shuffle their ratings, hold out 1 for test, 1 for val, rest train."""
    rng = np.random.RandomState(seed)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df['_rank'] = df.groupby('user_idx').cumcount()
    df['_count'] = df.groupby('user_idx')['user_idx'].transform('count')

    test = df[df['_rank'] == 0].copy()
    val = df[df['_rank'] == 1].copy()
    train = df[df['_rank'] >= 2].copy()
    return train.drop(columns=['_rank', '_count']), val.drop(columns=['_rank', '_count']), test.drop(columns=['_rank', '_count'])

train_df, val_df, test_df = leave_one_out_split(reviews)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Train: 18,171,881 | Val: 272,319 | Test: 272,319


## 4. Evaluation Harness

Shared across all models. Two regimes:

- **Rating prediction (RMSE)**: how accurately we predict the held-out rating, on the **full test set** (272K users).
- **Top-K ranking (Recall@10, NDCG@10)**: for each test user, rank their held-out game against 99 random unseen negatives (standard NCF-style protocol). Because our test set has 272K users, we **subsample** to `N_EVAL_USERS=10000` for ranking eval — this gives tight confidence intervals (~±0.5%) and keeps eval under 1 minute per model instead of 30+.

Set `N_EVAL_USERS = None` to evaluate on the full test set.

In [8]:
N_EVAL_USERS = 10000  # subsample for ranking eval; set to None for full test set

def rmse(preds, actuals):
    preds = np.asarray(preds); actuals = np.asarray(actuals)
    return float(np.sqrt(np.mean((preds - actuals) ** 2)))


def _sample_eval_subset(test_df, n=N_EVAL_USERS, seed=SEED):
    if n is None or n >= len(test_df):
        return test_df
    return test_df.sample(n=n, random_state=seed).reset_index(drop=True)


def _build_user_seen_arrays(train_df):
    """Build, for each user, a sorted numpy array of game_idx they've rated.

    Returns a dict {user_idx: np.ndarray}. Much faster for membership tests
    via np.isin than Python sets when used in the vectorized eval below.
    """
    user_seen = {}
    for u, grp in train_df.groupby('user_idx', sort=False)['game_idx']:
        user_seen[int(u)] = grp.values.astype(np.int32)
    return user_seen


def evaluate_ranking(model_score_fn, test_df, train_df, n_negatives=99, k=10, seed=SEED):
    """Vectorized ranking eval with subsampled test users.

    model_score_fn(user_idx:int, game_idx_array:np.ndarray) -> np.ndarray of scores
    (same length as game_idx_array, higher = better).
    """
    rng = np.random.RandomState(seed)
    eval_df = _sample_eval_subset(test_df)
    user_seen = _build_user_seen_arrays(train_df)

    # Oversample negatives per user, then filter out seen. Oversample by 1.5x to
    # almost always get enough on the first pass without per-user rejection loops.
    oversample = int(n_negatives * 1.5) + 5

    recalls, ndcgs = [], []
    users = eval_df['user_idx'].values.astype(np.int64)
    positives = eval_df['game_idx'].values.astype(np.int64)

    for u, pos in tqdm(zip(users, positives), total=len(users), desc="Ranking eval"):
        seen = user_seen.get(int(u), np.array([], dtype=np.int32))
        # Sample candidates, filter out anything the user has seen + the positive itself
        cand = rng.randint(0, N_GAMES, size=oversample)
        mask = ~np.isin(cand, seen) & (cand != pos)
        cand = cand[mask]
        if len(cand) < n_negatives:
            # Fallback: keep sampling until we have enough
            while len(cand) < n_negatives:
                extra = rng.randint(0, N_GAMES, size=oversample)
                extra = extra[~np.isin(extra, seen) & (extra != pos)]
                cand = np.concatenate([cand, extra])
        negs = cand[:n_negatives]
        candidates = np.concatenate([[pos], negs])
        scores = model_score_fn(int(u), candidates)
        # Rank of positive (at index 0 in candidates)
        rank = int((scores > scores[0]).sum())  # how many negatives beat it
        if rank < k:
            recalls.append(1.0)
            ndcgs.append(1.0 / np.log2(rank + 2))
        else:
            recalls.append(0.0)
            ndcgs.append(0.0)

    return {'recall@10': float(np.mean(recalls)),
            'ndcg@10': float(np.mean(ndcgs)),
            'n_eval': len(users)}

## 5. Model 1 — Baseline: Bayesian-Weighted Popularity

**Why not just average rating?** A game with 3 ratings of 10.0 would dominate over a game with 5000 ratings averaging 8.5. We shrink toward the global mean based on vote count — this is the IMDB Top 250 formula.

$$\text{score}(g) = \frac{v}{v + m} \cdot R_g + \frac{m}{v + m} \cdot C$$

Where $v$ = votes for game $g$, $R_g$ = mean rating for $g$, $C$ = global mean, $m$ = smoothing constant (we use the 75th percentile of vote counts).

In [9]:
class BayesianPopularity:
    def fit(self, train_df):
        self.C = train_df['rating'].mean()
        game_stats = train_df.groupby('game_idx')['rating'].agg(['mean', 'count']).reset_index()
        self.m = float(game_stats['count'].quantile(0.75))
        game_stats['score'] = (
            (game_stats['count'] / (game_stats['count'] + self.m)) * game_stats['mean']
            + (self.m / (game_stats['count'] + self.m)) * self.C
        )
        self.scores = np.full(N_GAMES, self.C, dtype=np.float32)
        self.scores[game_stats['game_idx'].values] = game_stats['score'].values
        return self

    def predict_rating(self, user_idx, game_idx):
        return self.scores[game_idx]

    def score(self, user_idx, game_idx_array):
        # Same score for everyone — it's a popularity model
        return self.scores[game_idx_array]


pop_model = BayesianPopularity().fit(train_df)

# Rating RMSE
preds = pop_model.predict_rating(test_df['user_idx'].values, test_df['game_idx'].values)
pop_rmse = rmse(preds, test_df['rating'].values)
print(f"Popularity RMSE: {pop_rmse:.4f}")

# Ranking
pop_rank = evaluate_ranking(pop_model.score, test_df, train_df)
print(f"Popularity Recall@10: {pop_rank['recall@10']:.4f}  NDCG@10: {pop_rank['ndcg@10']:.4f}")

Popularity RMSE: 1.4759


Ranking eval:   0%|          | 0/10000 [00:00<?, ?it/s]

Popularity Recall@10: 0.5159  NDCG@10: 0.3179


## 6. Model 2 — Basic: Content-Based Filtering

Build a TF-IDF vector for each game from its **mechanics, categories, designers, and description**. A user's taste profile is the weighted average of their rated games' content vectors (weighted by rating). Predict score = cosine similarity between user profile and candidate game.

This is mechanistically different from KNN (which uses rating co-occurrence) and SVD (which uses latent factors from the rating matrix). **It also works for cold-start games** — which matters for the novel cold-start contribution in the proposal.

In [10]:
# Build content features per game
def build_content_features(games_df, game_enc):
    valid_ids = set(game_enc.classes_)
    g = games_df[games_df['id'].isin(valid_ids)].copy()

    # Concatenate the metadata fields that describe a game's character
    text_fields = ['boardgamecategory', 'boardgamemechanic', 'boardgamefamily',
                   'boardgamedesigner', 'description']
    for f in text_fields:
        if f not in g.columns:
            g[f] = ''
        g[f] = g[f].fillna('').astype(str)

    # Mechanics/categories are stored as list-strings like "['Dice Rolling', 'Set Collection']"
    # Strip the list formatting and keep the words as tokens.
    def clean_list_str(s):
        return s.replace('[', ' ').replace(']', ' ').replace("'", ' ').replace('"', ' ').replace(',', ' ')

    for f in ['boardgamecategory', 'boardgamemechanic', 'boardgamefamily', 'boardgamedesigner']:
        g[f] = g[f].apply(clean_list_str)

    g['content'] = (
        (g['boardgamecategory'] + ' ') * 3 +   # upweight category
        (g['boardgamemechanic'] + ' ') * 3 +   # upweight mechanics
        g['boardgamefamily'] + ' ' +
        g['boardgamedesigner'] + ' ' +
        g['description']
    )

    # Map game_id -> game_idx so features align with our encoder
    g['game_idx'] = game_enc.transform(g['id'])
    g = g.sort_values('game_idx').reset_index(drop=True)

    # Some games may be missing from metadata; fill with empty string
    content_by_idx = pd.Series([''] * N_GAMES)
    content_by_idx.iloc[g['game_idx'].values] = g['content'].values
    return content_by_idx.tolist()


content_texts = build_content_features(games, game_enc)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2), min_df=5)
game_vectors = tfidf.fit_transform(content_texts)  # (N_GAMES, 5000)
print(f"Content matrix shape: {game_vectors.shape}")

Content matrix shape: (21802, 5000)


In [ ]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize


class ContentBased:
    """Content-based filter that builds user profiles lazily, per-user.

    Why lazy: materializing an (N_USERS, n_features) matrix for 272K users ×
    5000 TF-IDF features blows past Colab's 12 GB RAM because the product
    W @ game_vectors is not sparse enough to store cheaply. Instead we keep
    (1) per-user training rows indexed by CSR row pointers, and
    (2) the TF-IDF game matrix (already sparse & small).
    For each prediction we compute just that one user's profile on demand.
    """

    def __init__(self, game_vectors):
        self.game_vectors = game_vectors  # sparse (N_GAMES, F)

    def fit(self, train_df):
        # Per-user mean for centering (so we weight by "liked vs their baseline")
        self.user_means = train_df.groupby('user_idx')['rating'].mean()
        self.global_mean = float(train_df['rating'].mean())

        centered = (train_df['rating'].values
                    - train_df['user_idx'].map(self.user_means).values)

        # Build a (N_USERS, N_GAMES) CSR of centered ratings.
        # CSR gives us O(1) access to "which games did user u rate" via indptr,
        # which is what we need for the lazy profile computation.
        self.W = csr_matrix(
            (centered.astype(np.float32),
             (train_df['user_idx'].values, train_df['game_idx'].values)),
            shape=(N_USERS, N_GAMES),
        )

        # Pre-normalize game vectors so scoring is just a dot product.
        self.game_vectors_norm = normalize(self.game_vectors).tocsr()
        return self

    # ---- internal: build one user's profile on demand ----
    def _user_profile(self, u):
        """Return an L2-normalized 1 x F sparse row for user u."""
        row = self.W.getrow(u)                          # 1 x N_GAMES
        profile = row @ self.game_vectors_norm          # 1 x F  (usually sparse enough)
        # normalize; if the user has no training ratings we just return zeros
        n = np.sqrt(profile.multiply(profile).sum())
        if n > 0:
            profile = profile / n
        return profile

    # ---- batched helper for vectorized calls ----
    def _user_profiles_batch(self, user_ids):
        """Compute profiles for a batch of users as a sparse (B, F) matrix.

        Batching a few thousand users at a time is the sweet spot: the slice
        W[user_ids] is small, and its product with game_vectors_norm stays
        manageable in memory.
        """
        W_sub = self.W[user_ids]                        # B x N_GAMES
        profiles = W_sub @ self.game_vectors_norm      # B x F
        profiles = normalize(profiles)
        return profiles

    def score(self, user_idx, game_idx_array):
        # Single-user cosine scoring, used by evaluate_ranking.
        u_vec = self._user_profile(int(user_idx))      # 1 x F
        g_vecs = self.game_vectors_norm[game_idx_array]  # K x F
        sims = np.asarray((u_vec @ g_vecs.T).todense()).flatten()
        return sims

    def predict_rating(self, user_idx_arr, game_idx_arr):
        """Batched RMSE prediction: cosine(user_profile, game) mapped to rating."""
        u_idx = np.asarray(user_idx_arr)
        g_idx = np.asarray(game_idx_arr)
        out = np.empty(len(u_idx), dtype=np.float32)

        # Process in chunks so we never hold a giant dense matrix in RAM.
        CHUNK = 20000
        for start in tqdm(range(0, len(u_idx), CHUNK), desc="CB predict"):
            end = min(start + CHUNK, len(u_idx))
            users_chunk = u_idx[start:end]
            games_chunk = g_idx[start:end]

            # Unique users in this chunk -> compute their profiles once
            uniq_u, inv = np.unique(users_chunk, return_inverse=True)
            profiles = self._user_profiles_batch(uniq_u)    # (U_chunk, F) sparse
            game_rows = self.game_vectors_norm[games_chunk]  # (chunk, F) sparse

            # Pick the right profile row per (user, game) pair and dot it
            user_rows = profiles[inv]                       # (chunk, F) sparse
            sims = np.asarray(user_rows.multiply(game_rows).sum(axis=1)).flatten()

            means = (self.user_means.reindex(users_chunk)
                                     .fillna(self.global_mean).values)
            out[start:end] = np.clip(means + sims * 1.5, 1, 10)

        return out


cb_model = ContentBased(game_vectors).fit(train_df)

preds = cb_model.predict_rating(test_df['user_idx'].values, test_df['game_idx'].values)
cb_rmse = rmse(preds, test_df['rating'].values)
print(f"ContentBased RMSE: {cb_rmse:.4f}")

cb_rank = evaluate_ranking(cb_model.score, test_df, train_df)
print(f"ContentBased Recall@10: {cb_rank['recall@10']:.4f}  NDCG@10: {cb_rank['ndcg@10']:.4f}")


## 7. Model 3 — Advanced: Two-Tower Neural Network

Two separate MLPs (a "user tower" and a "game tower") each produce a D-dimensional embedding. Score = dot product. Trained with implicit feedback + negative sampling (BPR-style).

This is the architecture behind YouTube, Google Play, and most modern retrieval systems. Fast at inference (you can precompute all game embeddings once and do ANN lookup per user).

In [ ]:
# Implicit feedback dataset: each (user, positive_game) pair + k sampled negatives.
# Memory-efficient: we use rejection sampling WITHOUT a per-user positive set.
# At 18M rows and 22K games, the rejection probability is tiny (~0.3% per user on avg)
# so occasionally sampling a false negative is fine — standard BPR practice.
class BPRDataset(Dataset):
    def __init__(self, train_df, n_games, n_neg=4, seed=SEED):
        self.users = train_df['user_idx'].values.astype(np.int64)
        self.pos = train_df['game_idx'].values.astype(np.int64)
        self.n_games = n_games
        self.n_neg = n_neg
        # Use numpy's default_rng which is fork-safe and fast
        self.seed = seed

    def __len__(self):
        return len(self.users)

    def __getitem__(self, i):
        # Fast path: sample n_neg randoms, make sure none equals the positive.
        # We don't reject all the user's positives (too expensive at scale).
        negs = np.random.randint(0, self.n_games, size=self.n_neg).astype(np.int64)
        pos = self.pos[i]
        # Resample any collisions with the positive
        mask = (negs == pos)
        while mask.any():
            negs[mask] = np.random.randint(0, self.n_games, size=mask.sum())
            mask = (negs == pos)
        return self.users[i], pos, negs


class TwoTower(nn.Module):
    def __init__(self, n_users, n_games, emb_dim=64, tower_dim=64, dropout=0.2):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.game_emb = nn.Embedding(n_games, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.game_emb.weight, std=0.01)

        self.user_tower = nn.Sequential(
            nn.Linear(emb_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, tower_dim)
        )
        self.game_tower = nn.Sequential(
            nn.Linear(emb_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, tower_dim)
        )

    def user_forward(self, u):
        return self.user_tower(self.user_emb(u))

    def game_forward(self, g):
        return self.game_tower(self.game_emb(g))

    def forward(self, u, g):
        return (self.user_forward(u) * self.game_forward(g)).sum(-1)

In [ ]:
# Train
EMB_DIM = 64
TOWER_DIM = 64
BATCH = 8192        # bigger on GPU; drop to 2048 if OOM
EPOCHS = 5          # 18M rows/epoch is a lot — 5 epochs usually plenty for BPR
LR = 1e-3
N_NEG = 4

ds = BPRDataset(train_df, N_GAMES, n_neg=N_NEG)
# num_workers=0 avoids forking a dict-of-arrays copy per worker process
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=(DEVICE.type=='cuda'))

model = TwoTower(N_USERS, N_GAMES, EMB_DIM, TOWER_DIM).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-6)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    n_seen = 0
    pbar = tqdm(dl, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for u, p, negs in pbar:
        u = u.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        negs = negs.to(DEVICE, non_blocking=True)
        u_vec = model.user_forward(u)                    # (B, D)
        p_vec = model.game_forward(p)                    # (B, D)
        n_vec = model.game_forward(negs.view(-1)).view(negs.size(0), negs.size(1), -1)  # (B, K, D)
        pos_score = (u_vec * p_vec).sum(-1, keepdim=True)  # (B, 1)
        neg_score = (u_vec.unsqueeze(1) * n_vec).sum(-1)   # (B, K)
        # BPR loss: -log sigmoid(pos - neg)
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * u.size(0)
        n_seen += u.size(0)
        if n_seen % (BATCH * 50) == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    print(f"Epoch {epoch+1}: avg loss = {total_loss / n_seen:.4f}")

In [ ]:
# Precompute all game embeddings once for fast scoring
model.eval()
with torch.no_grad():
    all_games = torch.arange(N_GAMES, device=DEVICE)
    game_embs = model.game_forward(all_games).cpu().numpy()  # (N_GAMES, TOWER_DIM)

    all_users = torch.arange(N_USERS, device=DEVICE)
    # User embeddings in chunks to avoid OOM
    user_embs = []
    for i in range(0, N_USERS, 10000):
        chunk = all_users[i:i+10000]
        user_embs.append(model.user_forward(chunk).cpu().numpy())
    user_embs = np.concatenate(user_embs, axis=0)  # (N_USERS, TOWER_DIM)

print(f"User emb: {user_embs.shape} | Game emb: {game_embs.shape}")

In [ ]:
class TwoTowerScorer:
    def __init__(self, user_embs, game_embs, train_df):
        self.U = user_embs
        self.G = game_embs
        # For rating prediction fallback: user mean from train
        self.user_means = train_df.groupby('user_idx')['rating'].mean()
        self.global_mean = train_df['rating'].mean()

    def score(self, user_idx, game_idx_array):
        return self.U[user_idx] @ self.G[game_idx_array].T

    def predict_rating(self, user_idx_arr, game_idx_arr):
        """Vectorized: tanh-squashed score anchored at user's mean."""
        u_idx = np.asarray(user_idx_arr)
        g_idx = np.asarray(game_idx_arr)
        scores = (self.U[u_idx] * self.G[g_idx]).sum(axis=1)
        means = self.user_means.reindex(u_idx).fillna(self.global_mean).values
        devs = np.tanh(scores * 0.5) * 1.5
        return np.clip(means + devs, 1, 10)


tt = TwoTowerScorer(user_embs, game_embs, train_df)

preds = tt.predict_rating(test_df['user_idx'].values, test_df['game_idx'].values)
tt_rmse = rmse(preds, test_df['rating'].values)
print(f"TwoTower RMSE: {tt_rmse:.4f}")

tt_rank = evaluate_ranking(tt.score, test_df, train_df)
print(f"TwoTower Recall@10: {tt_rank['recall@10']:.4f}  NDCG@10: {tt_rank['ndcg@10']:.4f}")

## 8. Results Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'Popularity (Bayesian)', 'Type': 'Baseline', 'RMSE': pop_rmse, 'Recall@10': pop_rank['recall@10'], 'NDCG@10': pop_rank['ndcg@10']},
    {'Model': 'Content-Based (TF-IDF)', 'Type': 'Basic', 'RMSE': cb_rmse, 'Recall@10': cb_rank['recall@10'], 'NDCG@10': cb_rank['ndcg@10']},
    {'Model': 'Two-Tower NN', 'Type': 'Advanced', 'RMSE': tt_rmse, 'Recall@10': tt_rank['recall@10'], 'NDCG@10': tt_rank['ndcg@10']},
])
print(results.to_string(index=False))

## 9. Save splits for teammates

Alex and Brandon should use the **same train/val/test split** so all six models are directly comparable.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/bgg_recsys' if IN_COLAB and os.path.exists('/content/drive/MyDrive') else './bgg_splits'
os.makedirs(OUT_DIR, exist_ok=True)

train_df.to_parquet(f'{OUT_DIR}/train.parquet', index=False)
val_df.to_parquet(f'{OUT_DIR}/val.parquet', index=False)
test_df.to_parquet(f'{OUT_DIR}/test.parquet', index=False)

# Save encoders too
import pickle
with open(f'{OUT_DIR}/encoders.pkl', 'wb') as f:
    pickle.dump({'user_enc': user_enc, 'game_enc': game_enc,
                 'N_USERS': N_USERS, 'N_GAMES': N_GAMES}, f)

print(f"Saved splits to {OUT_DIR}")
print("Alex/Brandon: load with pd.read_parquet() and use the same evaluate_ranking() function.")